
# PathSnap — GPS Point Route-Label Generation

**Purpose:** Convert each trip's chronological `.route` arc sequence into one route-arc label per GPS observation in `.track`.

This notebook is designed for the PathSnap dataset structure used by `01_Dataset_Exploration.ipynb`.

## What this notebook does

1. Loads `.track`, `.nodes`, `.arcs`, and `.route` files with strict column/type/range checks.
2. Builds the ordered route geometry using the `.route` arc IDs (not the row order of `.arcs`).
3. Projects longitude/latitude into a local metric coordinate system before measuring distances.
4. Assigns GPS observations to route arcs using **global monotonic sequence alignment**:
   - GPS points are processed in timestamp order.
   - A point can stay on the same route position or move forward.
   - A point can never move backward along the supplied route sequence.
   - The same physical `arc_id` may occur more than once; `route_position` distinguishes those visits.
5. Keeps unmatched/invalid GPS observations visible in the output instead of silently dropping them.
6. Saves per-point labels separately for `train` and `student_test`, plus summaries and a Folium inspection map.

## Important data-use rules

- `.track`: longitude, latitude, timestamp (seconds), no header.
- `.nodes`: longitude, latitude, no header. The node ID is its **zero-based row index**.
- `.arcs`: from-node ID, to-node ID, no header. The arc ID is its **zero-based row index**.
- `.route`: chronological sequence of arc IDs, no header.
- GPS rows do **not** correspond one-to-one with `.route` rows.
- This notebook deliberately processes only `train` and `student_test`. It does not open the evaluator-testing partition.
- `student_test` labels are for development-stage validation/comparison only. Do not use them to fit the model or tune model parameters. Train models using `train` labels only.
- The route sequence is treated as the ordered reference path. This algorithm does not claim that every GPS point is guaranteed to be close to the route; it reports distance and match status so questionable labels can be inspected.

Run all cells from top to bottom. Review the validation summary and map before sharing labels with teammates.


In [ ]:

# 1. Google Drive and project paths
from pathlib import Path

if Path("/content/drive/MyDrive").exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive")
elif Path("/content/gdrive/MyDrive").exists():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")
else:
    from google.colab import drive
    drive.mount("/content/gdrive")
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

PROJECT_DIR = DRIVE_ROOT / "PathSnap"
LABEL_DIR = PROJECT_DIR / "results" / "labels"
MAP_DIR = PROJECT_DIR / "results" / "maps" / "label_checks"

LABEL_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Label output directory:", LABEL_DIR)


In [ ]:

# 2. Install/import required packages
%pip -q install numpy pandas pyproj folium matplotlib

import math
import warnings
import numpy as np
import pandas as pd
import folium
import matplotlib.pyplot as plt

from pyproj import CRS, Transformer
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Maximum cells in the exact dynamic-programming alignment matrix.
# If a trip exceeds this limit, the notebook stops rather than risking a Colab
# out-of-memory crash. Increase only if the runtime has sufficient RAM.
MAX_DP_CELLS = 25_000_000

# A point farther than this distance from its assigned route segment is flagged
# as unmatched. Its best route position is still retained for diagnosis, but
# `is_matched` is False and downstream training should exclude it.
MATCH_DISTANCE_THRESHOLD_M = 100.0

# Stop on disconnected consecutive route arcs. This avoids producing apparently
# valid labels from a malformed route. Change only after inspecting the trip.
STRICT_ROUTE_CONTINUITY = True

# Save CSV always. Parquet is also saved when a parquet engine is available.
SAVE_PARQUET = True


In [ ]:

# 3. Locate the extracted dataset
# Expected:
# PathSnap/data/student_dataset/Student folder/
#     train/<trip_id>/
#     student_test/<trip_id>/

possible_roots = [
    PROJECT_DIR / "data" / "student_dataset" / "Student folder",
    PROJECT_DIR / "data" / "raw" / "student_dataset" / "Student folder",
    PROJECT_DIR / "data" / "student_dataset",
    PROJECT_DIR / "data" / "raw" / "student_dataset",
]

DATA_DIR = None
for candidate in possible_roots:
    if (candidate / "train").is_dir() and (candidate / "student_test").is_dir():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    search_base = PROJECT_DIR / "data"
    if search_base.exists():
        for train_dir in search_base.rglob("train"):
            parent = train_dir.parent
            if (parent / "student_test").is_dir():
                DATA_DIR = parent
                break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find both 'train' and 'student_test' directories. "
        "Check that the dataset ZIP is extracted under PathSnap/data/."
    )

print("Dataset directory:", DATA_DIR)
print("Train trips:", len([p for p in (DATA_DIR / "train").iterdir() if p.is_dir()]))
print("Student-test trips:", len([p for p in (DATA_DIR / "student_test").iterdir() if p.is_dir()]))
print("Evaluator partition is intentionally not loaded.")


In [ ]:

# 4. Strict file readers and trip loader
FILE_COLUMNS = {
    "track": ["longitude", "latitude", "timestamp"],
    "nodes": ["longitude", "latitude"],
    "arcs": ["from_node", "to_node"],
    "route": ["arc_id"],
}

def read_exact_whitespace_file(path: Path, expected_columns: list[str]) -> pd.DataFrame:
    """Read a headerless whitespace file and enforce its exact column count."""
    if not path.is_file():
        raise FileNotFoundError(f"Required file is missing: {path}")

    # Read without assigning names first, so extra/missing columns are detected.
    raw = pd.read_csv(path, sep=r"\s+", header=None, dtype=str, engine="python")
    if raw.shape[1] != len(expected_columns):
        raise ValueError(
            f"{path.name}: expected {len(expected_columns)} columns "
            f"{expected_columns}, found {raw.shape[1]} columns."
        )

    raw.columns = expected_columns
    for col in expected_columns:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

    if raw.isna().any().any():
        bad = raw.isna().sum()
        raise ValueError(f"{path.name}: non-numeric or missing values detected: {bad[bad > 0].to_dict()}")

    return raw

def load_trip(trip_folder: Path) -> dict:
    trip_id = trip_folder.name
    data = {}
    for suffix, columns in FILE_COLUMNS.items():
        path = trip_folder / f"{trip_id}.{suffix}"
        data[suffix] = read_exact_whitespace_file(path, columns)
    return data

def list_trip_folders(partition: str) -> list[Path]:
    if partition not in ("train", "student_test"):
        raise ValueError("Only 'train' and 'student_test' are permitted in this notebook.")
    folder = DATA_DIR / partition
    if not folder.is_dir():
        raise FileNotFoundError(f"Partition folder not found: {folder}")
    return sorted(p for p in folder.iterdir() if p.is_dir())

print("Loader ready.")


In [ ]:

# 5. Validate node, arc, route and GPS indexing
def validate_trip_data(data: dict, trip_id: str) -> dict:
    track, nodes, arcs, route = (data[k].copy() for k in ("track", "nodes", "arcs", "route"))

    # GPS coordinate validity is recorded per observation rather than dropping rows.
    gps_valid = (
        np.isfinite(track["longitude"].to_numpy(dtype=float)) &
        np.isfinite(track["latitude"].to_numpy(dtype=float)) &
        track["longitude"].between(-180, 180).to_numpy() &
        track["latitude"].between(-90, 90).to_numpy()
    )

    # Nodes are identified by their row positions: 0, 1, ..., len(nodes)-1.
    node_lon = nodes["longitude"].to_numpy(dtype=float)
    node_lat = nodes["latitude"].to_numpy(dtype=float)
    nodes_valid = (
        np.isfinite(node_lon) & np.isfinite(node_lat) &
        (node_lon >= -180) & (node_lon <= 180) &
        (node_lat >= -90) & (node_lat <= 90)
    )
    if not nodes_valid.all():
        raise ValueError(f"{trip_id}: {int((~nodes_valid).sum())} road nodes have invalid coordinates.")

    # Arc IDs are their row positions; endpoints must refer to valid node row IDs.
    arc_from_num = arcs["from_node"].to_numpy(dtype=float)
    arc_to_num = arcs["to_node"].to_numpy(dtype=float)
    if not np.equal(arc_from_num, np.floor(arc_from_num)).all():
        raise ValueError(f"{trip_id}: .arcs contains non-integer from_node IDs.")
    if not np.equal(arc_to_num, np.floor(arc_to_num)).all():
        raise ValueError(f"{trip_id}: .arcs contains non-integer to_node IDs.")

    arc_from = arc_from_num.astype(np.int64)
    arc_to = arc_to_num.astype(np.int64)
    arc_endpoints_valid = (
        (arc_from >= 0) & (arc_from < len(nodes)) &
        (arc_to >= 0) & (arc_to < len(nodes))
    )
    if not arc_endpoints_valid.all():
        bad_rows = np.flatnonzero(~arc_endpoints_valid)[:10].tolist()
        raise ValueError(f"{trip_id}: invalid node references in .arcs rows (arc IDs): {bad_rows}")

    route_num = route["arc_id"].to_numpy(dtype=float)
    if not np.equal(route_num, np.floor(route_num)).all():
        raise ValueError(f"{trip_id}: .route contains non-integer arc IDs.")
    route_ids = route_num.astype(np.int64)
    route_ids_valid = (route_ids >= 0) & (route_ids < len(arcs))
    if not route_ids_valid.all():
        bad_positions = np.flatnonzero(~route_ids_valid)[:10].tolist()
        raise ValueError(f"{trip_id}: route contains out-of-range arc IDs at route positions {bad_positions}")

    if len(route_ids) == 0:
        raise ValueError(f"{trip_id}: .route is empty.")
    if len(nodes) == 0 or len(arcs) == 0:
        raise ValueError(f"{trip_id}: road network has no nodes or arcs.")

    route_from = arc_from[route_ids]
    route_to = arc_to[route_ids]
    continuity = route_to[:-1] == route_from[1:] if len(route_ids) > 1 else np.array([], dtype=bool)
    continuity_breaks = int((~continuity).sum())

    # Timestamp order is reported. The labeler uses stable timestamp order.
    timestamps = track["timestamp"].to_numpy(dtype=float)
    timestamp_valid = np.isfinite(timestamps)
    timestamp_monotonic_in_file = bool(
        timestamp_valid.all() and np.all(np.diff(timestamps) >= 0)
    )

    return {
        "gps_valid_mask": gps_valid,
        "route_ids": route_ids,
        "route_from": route_from,
        "route_to": route_to,
        "route_continuity_breaks": continuity_breaks,
        "route_is_continuous": continuity_breaks == 0,
        "timestamp_monotonic_in_file": timestamp_monotonic_in_file,
        "invalid_gps_count": int((~gps_valid).sum()),
    }

print("Validation function ready.")


In [ ]:

# 6. Create a local metric projection
def make_local_transformer(longitudes, latitudes):
    """
    Build a local azimuthal-equidistant projection centered on the trip.
    This provides metre-based distances without measuring in raw degrees.
    """
    lon0 = float(np.nanmean(longitudes))
    lat0 = float(np.nanmean(latitudes))
    local_crs = CRS.from_proj4(
        f"+proj=aeqd +lat_0={lat0:.10f} +lon_0={lon0:.10f} "
        "+datum=WGS84 +units=m +no_defs"
    )
    transformer = Transformer.from_crs("EPSG:4326", local_crs, always_xy=True)
    return transformer, local_crs.to_string()

def point_to_segment_distances(px, py, ax, ay, bx, by):
    """Return Euclidean distances (metres) from one point to every route segment."""
    vx = bx - ax
    vy = by - ay
    denom = vx * vx + vy * vy

    # Projection fraction of point onto each finite segment, clipped to [0, 1].
    with np.errstate(divide="ignore", invalid="ignore"):
        t = ((px - ax) * vx + (py - ay) * vy) / denom
    t = np.where(denom > 0, t, 0.0)
    t = np.clip(t, 0.0, 1.0)

    closest_x = ax + t * vx
    closest_y = ay + t * vy
    return np.hypot(px - closest_x, py - closest_y), t



## 7. Label-generation method: monotonic dynamic programming

For every valid GPS point \(i\) and route position \(j\), calculate the point-to-segment distance \(d(i,j)\) in metres.

The dynamic-programming recurrence is:

\[
C(i,j) = d(i,j)^2 + \min_{k \leq j} C(i-1,k)
\]

The prefix minimum enforces the non-decreasing route-position constraint. Backtracking recovers the lowest-cost sequence across the entire trip, rather than independently choosing the nearest arc for each point.

**Why this matters:** independent nearest-arc matching can jump backward when roads are close or parallel. The monotonic constraint prevents a GPS observation later in time from being assigned to an earlier position in the supplied route.

A repeated physical `arc_id` is not ambiguous here: `route_position` records which occurrence in the route sequence was selected.


In [ ]:

# 8. Exact monotonic sequence aligner
def monotonic_route_alignment(gps_x, gps_y, seg_ax, seg_ay, seg_bx, seg_by,
                              max_dp_cells=MAX_DP_CELLS):
    """
    Globally align GPS points to an ordered list of route segments.

    Returns:
      route_positions: index into the ordered .route sequence for each GPS point
      distances_m: distance from each GPS point to its selected segment
      fractions: position along the selected segment (0=start, 1=end)

    Constraint: route_positions[i] >= route_positions[i-1].
    Repeated route positions are allowed.
    """
    n_points = len(gps_x)
    n_route = len(seg_ax)

    if n_points == 0:
        return np.array([], dtype=int), np.array([], dtype=float), np.array([], dtype=float)
    if n_route == 0:
        raise ValueError("Cannot align GPS points to an empty route.")
    if n_points * n_route > max_dp_cells:
        raise MemoryError(
            f"Exact alignment requires {n_points*n_route:,} DP cells "
            f"({n_points} GPS points x {n_route} route arcs), exceeding "
            f"MAX_DP_CELLS={max_dp_cells:,}. Do not continue blindly: "
            "inspect trip size/runtime RAM before increasing MAX_DP_CELLS."
        )

    backpointer = np.zeros((n_points, n_route), dtype=np.int32)
    prev_cost = None
    final_distances = None
    final_fractions = None

    for i in range(n_points):
        distances, fractions = point_to_segment_distances(
            gps_x[i], gps_y[i], seg_ax, seg_ay, seg_bx, seg_by
        )
        point_cost = distances * distances

        if i == 0:
            current_cost = point_cost
            backpointer[i, :] = 0
        else:
            # Prefix minimum: best previous route position among positions <= j.
            prefix_best_idx = np.empty(n_route, dtype=np.int32)
            prefix_best_cost = np.empty(n_route, dtype=float)

            best_idx = 0
            best_cost = prev_cost[0]
            for j in range(n_route):
                if prev_cost[j] < best_cost:
                    best_cost = prev_cost[j]
                    best_idx = j
                prefix_best_idx[j] = best_idx
                prefix_best_cost[j] = best_cost

            current_cost = point_cost + prefix_best_cost
            backpointer[i, :] = prefix_best_idx

        prev_cost = current_cost
        final_distances = distances
        final_fractions = fractions

    # Backtrack from the best ending route position.
    route_positions = np.empty(n_points, dtype=np.int32)
    route_positions[-1] = int(np.argmin(prev_cost))
    for i in range(n_points - 1, 0, -1):
        route_positions[i - 1] = backpointer[i, route_positions[i]]

    # Recompute selected distances/fractions for clarity and avoid retaining
    # an N x M distance matrix.
    selected_distances = np.empty(n_points, dtype=float)
    selected_fractions = np.empty(n_points, dtype=float)
    for i, j in enumerate(route_positions):
        d, t = point_to_segment_distances(
            gps_x[i], gps_y[i],
            seg_ax[j:j+1], seg_ay[j:j+1], seg_bx[j:j+1], seg_by[j:j+1]
        )
        selected_distances[i] = d[0]
        selected_fractions[i] = t[0]

    return route_positions, selected_distances, selected_fractions

print("Monotonic alignment function ready.")


In [ ]:

# 9. Generate labels for one trip
def generate_trip_labels(trip_folder: Path, partition: str):
    trip_id = trip_folder.name
    data = load_trip(trip_folder)
    validation = validate_trip_data(data, trip_id)

    if STRICT_ROUTE_CONTINUITY and not validation["route_is_continuous"]:
        raise ValueError(
            f"{trip_id}: route has {validation['route_continuity_breaks']} "
            "consecutive-arc continuity break(s). Inspect the route before labeling. "
            "If the dataset documentation permits this, set STRICT_ROUTE_CONTINUITY=False "
            "and clearly report the issue."
        )

    track = data["track"].copy()
    nodes = data["nodes"]
    route_ids = validation["route_ids"]
    route_from = validation["route_from"]
    route_to = validation["route_to"]

    # Retain the original .track row number for exact traceability.
    track["gps_row_index"] = np.arange(len(track), dtype=np.int64)
    track["gps_valid"] = validation["gps_valid_mask"]

    # The route's ordered arc geometry, not all network arcs.
    node_lon = nodes["longitude"].to_numpy(dtype=float)
    node_lat = nodes["latitude"].to_numpy(dtype=float)
    route_start_lon = node_lon[route_from]
    route_start_lat = node_lat[route_from]
    route_end_lon = node_lon[route_to]
    route_end_lat = node_lat[route_to]

    valid_gps = track["gps_valid"].to_numpy()
    if valid_gps.sum() == 0:
        raise ValueError(f"{trip_id}: no valid GPS coordinates available for alignment.")

    valid_lon = track.loc[valid_gps, "longitude"].to_numpy(dtype=float)
    valid_lat = track.loc[valid_gps, "latitude"].to_numpy(dtype=float)

    # Center projection on both observed GPS and route geometry.
    all_lon = np.concatenate([valid_lon, route_start_lon, route_end_lon])
    all_lat = np.concatenate([valid_lat, route_start_lat, route_end_lat])
    transformer, projection_name = make_local_transformer(all_lon, all_lat)

    gps_x, gps_y = transformer.transform(valid_lon, valid_lat)
    start_x, start_y = transformer.transform(route_start_lon, route_start_lat)
    end_x, end_y = transformer.transform(route_end_lon, route_end_lat)

    # Stable timestamp order. Invalid timestamps are excluded from alignment
    # and remain in the output as unmatched rows.
    valid_indices = np.flatnonzero(valid_gps)
    valid_timestamps = track.loc[valid_gps, "timestamp"].to_numpy(dtype=float)
    timestamp_ok = np.isfinite(valid_timestamps)

    alignable_local = np.flatnonzero(timestamp_ok)
    if len(alignable_local) == 0:
        raise ValueError(f"{trip_id}: valid GPS coordinates exist, but no valid timestamps exist.")

    # Stable sort preserves source row order for equal timestamps.
    order = np.argsort(valid_timestamps[alignable_local], kind="stable")
    sorted_local = alignable_local[order]
    sorted_track_positions = valid_indices[sorted_local]

    route_positions_sorted, distances_sorted, fractions_sorted = monotonic_route_alignment(
        gps_x[sorted_local], gps_y[sorted_local],
        start_x, start_y, end_x, end_y
    )

    # Map alignment results back to the original GPS row order.
    n = len(track)
    route_position = np.full(n, -1, dtype=np.int64)
    distance_m = np.full(n, np.nan, dtype=float)
    fraction_along_arc = np.full(n, np.nan, dtype=float)

    route_position[sorted_track_positions] = route_positions_sorted
    distance_m[sorted_track_positions] = distances_sorted
    fraction_along_arc[sorted_track_positions] = fractions_sorted

    assigned = route_position >= 0
    route_arc_id = np.full(n, -1, dtype=np.int64)
    route_arc_id[assigned] = route_ids[route_position[assigned]]

    from_node_id = np.full(n, -1, dtype=np.int64)
    to_node_id = np.full(n, -1, dtype=np.int64)
    from_node_id[assigned] = route_from[route_position[assigned]]
    to_node_id[assigned] = route_to[route_position[assigned]]

    is_matched = assigned & (distance_m <= MATCH_DISTANCE_THRESHOLD_M)
    status = np.full(n, "invalid_gps_or_timestamp", dtype=object)
    status[assigned] = "beyond_distance_threshold"
    status[is_matched] = "matched"

    labels = track.copy()
    labels["partition"] = partition
    labels["trip_id"] = trip_id
    labels["route_position"] = route_position
    labels["arc_id"] = route_arc_id
    labels["from_node_id"] = from_node_id
    labels["to_node_id"] = to_node_id
    labels["fraction_along_arc"] = fraction_along_arc
    labels["distance_to_assigned_arc_m"] = distance_m
    labels["is_matched"] = is_matched
    labels["label_status"] = status
    labels["projection"] = projection_name

    # A compact, explicit warning if timestamps were not monotonic in file order.
    summary = {
        "partition": partition,
        "trip_id": trip_id,
        "gps_rows": len(track),
        "valid_gps_rows": int(valid_gps.sum()),
        "invalid_gps_rows": int((~valid_gps).sum()),
        "valid_timestamp_rows": int(np.isfinite(track["timestamp"].to_numpy(dtype=float)).sum()),
        "route_positions": len(route_ids),
        "route_continuity_breaks": validation["route_continuity_breaks"],
        "timestamp_monotonic_in_file": validation["timestamp_monotonic_in_file"],
        "matched_rows": int(is_matched.sum()),
        "unmatched_rows": int((~is_matched).sum()),
        "matched_fraction_of_all_gps": float(is_matched.mean()) if n else np.nan,
        "median_assigned_distance_m": float(np.nanmedian(distance_m)) if np.isfinite(distance_m).any() else np.nan,
        "p95_assigned_distance_m": float(np.nanpercentile(distance_m[np.isfinite(distance_m)], 95))
            if np.isfinite(distance_m).any() else np.nan,
        "max_assigned_distance_m": float(np.nanmax(distance_m)) if np.isfinite(distance_m).any() else np.nan,
        "projection": projection_name,
        "match_threshold_m": MATCH_DISTANCE_THRESHOLD_M,
    }

    return labels, summary

print("Per-trip label generator ready.")



## 10. Run label generation

The next cell processes every trip in `train` and `student_test` only. It writes separate output files, so the development labels cannot accidentally be mixed into the training labels.

If one trip fails validation, it is recorded in the error table and the remaining trips continue. Do not ignore failures: inspect them before giving the labels to teammates.


In [ ]:

# 10. Process permitted partitions
all_summaries = []
all_errors = []

for partition in ("train", "student_test"):
    folders = list_trip_folders(partition)
    print(f"\\nProcessing {partition}: {len(folders)} trips")

    partition_frames = []
    for idx, trip_folder in enumerate(folders, start=1):
        try:
            labels, summary = generate_trip_labels(trip_folder, partition)
            partition_frames.append(labels)
            all_summaries.append(summary)
            print(
                f"[{idx:>3}/{len(folders)}] {trip_folder.name}: "
                f"{summary['matched_rows']}/{summary['gps_rows']} matched "
                f"({summary['matched_fraction_of_all_gps']:.1%})"
            )
        except Exception as exc:
            all_errors.append({
                "partition": partition,
                "trip_id": trip_folder.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            })
            print(f"[{idx:>3}/{len(folders)}] {trip_folder.name}: FAILED — {exc}")

    if partition_frames:
        partition_labels = pd.concat(partition_frames, ignore_index=True)
        csv_path = LABEL_DIR / f"{partition}_gps_route_labels.csv"
        partition_labels.to_csv(csv_path, index=False)
        print("Saved:", csv_path, "| rows:", len(partition_labels))

        if SAVE_PARQUET:
            parquet_path = LABEL_DIR / f"{partition}_gps_route_labels.parquet"
            try:
                partition_labels.to_parquet(parquet_path, index=False)
                print("Saved:", parquet_path)
            except (ImportError, ValueError) as exc:
                print("Parquet not saved (optional dependency missing):", exc)
    else:
        print(f"No successful labels generated for {partition}.")

summary_df = pd.DataFrame(all_summaries)
errors_df = pd.DataFrame(all_errors)

if not summary_df.empty:
    summary_path = LABEL_DIR / "label_generation_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print("\\nSaved summary:", summary_path)
    display(summary_df)

if not errors_df.empty:
    error_path = LABEL_DIR / "label_generation_errors.csv"
    errors_df.to_csv(error_path, index=False)
    print("\\nSome trips failed. Error report:", error_path)
    display(errors_df)
else:
    print("\\nNo trip-level errors.")


In [ ]:

# 11. Label quality checks
if not summary_df.empty:
    print("Overall label-generation summary")
    total_gps = int(summary_df["gps_rows"].sum())
    total_matched = int(summary_df["matched_rows"].sum())
    total_unmatched = int(summary_df["unmatched_rows"].sum())

    print("Trips processed:", len(summary_df))
    print("GPS rows:", total_gps)
    print("Matched rows:", total_matched)
    print("Unmatched rows:", total_unmatched)
    print("Overall matched fraction:", round(total_matched / total_gps, 4) if total_gps else np.nan)

    print("\\nTrips with less than 90% matched GPS rows:")
    display(summary_df.loc[
        summary_df["matched_fraction_of_all_gps"] < 0.90,
        ["partition", "trip_id", "matched_fraction_of_all_gps",
         "median_assigned_distance_m", "p95_assigned_distance_m",
         "route_continuity_breaks", "timestamp_monotonic_in_file"]
    ])

    print("\\nRoute-position monotonicity is guaranteed by the alignment algorithm for valid-timestamp rows.")
    print("Rows marked unmatched should not be used as training examples without review.")
else:
    print("No successful trip labels exist. Resolve errors before continuing.")



## 12. Inspect one labelled trip on a Folium map

The map displays:
- Observed GPS trace.
- Ordered ground-truth route geometry reconstructed from `.route`.
- GPS points assigned to route positions, coloured by match status.
- Start/end markers and layer controls.

This is an inspection map, not proof that every label is correct. Open several trips, especially trips with low match rates or large 95th-percentile distances.


In [ ]:

# 12. Build a visual QA map for one successful training trip
def build_label_check_map(trip_folder: Path, labels: pd.DataFrame):
    data = load_trip(trip_folder)
    validation = validate_trip_data(data, trip_folder.name)

    track = data["track"].copy()
    nodes = data["nodes"]
    route_ids = validation["route_ids"]
    route_from = validation["route_from"]
    route_to = validation["route_to"]

    gps_ok = (
        track["longitude"].between(-180, 180) &
        track["latitude"].between(-90, 90)
    )
    gps = track.loc[gps_ok].copy()
    if gps.empty:
        raise ValueError("No valid GPS coordinates to display.")

    center = [float(gps["latitude"].mean()), float(gps["longitude"].mean())]
    m = folium.Map(location=center, zoom_start=14, control_scale=True)

    # GPS trace.
    gps_coords = gps[["latitude", "longitude"]].to_numpy().tolist()
    folium.PolyLine(
        gps_coords, color="blue", weight=3, opacity=0.75,
        tooltip="Observed GPS trace"
    ).add_to(m)

    # Route polyline reconstructed from ordered route arcs.
    node_lon = nodes["longitude"].to_numpy(dtype=float)
    node_lat = nodes["latitude"].to_numpy(dtype=float)
    route_coords = [[float(node_lat[route_from[0]]), float(node_lon[route_from[0]])]]
    route_coords.extend([[float(node_lat[nid]), float(node_lon[nid])] for nid in route_to])

    folium.PolyLine(
        route_coords, color="orange", weight=5, opacity=0.85,
        tooltip="Ground-truth route from ordered .route arc IDs"
    ).add_to(m)

    # Matched and unmatched labelled observations.
    trip_labels = labels[labels["trip_id"] == trip_folder.name].copy()
    matched = trip_labels[trip_labels["is_matched"]]
    unmatched = trip_labels[~trip_labels["is_matched"]]

    matched_group = folium.FeatureGroup(name="Matched GPS observations", show=True)
    for row in matched.itertuples():
        folium.CircleMarker(
            location=[row.latitude, row.longitude],
            radius=3, color="green", fill=True, fill_opacity=0.75,
            tooltip=f"GPS row {row.gps_row_index} | arc_id {row.arc_id} | "
                    f"route_position {row.route_position} | {row.distance_to_assigned_arc_m:.1f} m"
        ).add_to(matched_group)
    matched_group.add_to(m)

    unmatched_group = folium.FeatureGroup(name="Unmatched / beyond threshold", show=True)
    for row in unmatched.itertuples():
        if pd.notna(row.latitude) and pd.notna(row.longitude):
            if -90 <= row.latitude <= 90 and -180 <= row.longitude <= 180:
                folium.CircleMarker(
                    location=[row.latitude, row.longitude],
                    radius=4, color="red", fill=True, fill_opacity=0.8,
                    tooltip=f"GPS row {row.gps_row_index} | status {row.label_status} | "
                            f"distance {row.distance_to_assigned_arc_m}"
                ).add_to(unmatched_group)
    unmatched_group.add_to(m)

    folium.Marker(gps_coords[0], tooltip="GPS start", icon=folium.Icon(color="green")).add_to(m)
    folium.Marker(gps_coords[-1], tooltip="GPS end", icon=folium.Icon(color="red")).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    return m

# Choose first successfully generated training trip.
successful_train_ids = (
    summary_df.loc[summary_df["partition"] == "train", "trip_id"].tolist()
    if not summary_df.empty else []
)

if successful_train_ids:
    sample_trip_id = successful_train_ids[0]
    sample_folder = DATA_DIR / "train" / sample_trip_id
    sample_labels = pd.read_csv(LABEL_DIR / "train_gps_route_labels.csv")

    qa_map = build_label_check_map(sample_folder, sample_labels)
    map_path = MAP_DIR / f"label_check_{sample_trip_id}.html"
    qa_map.save(str(map_path))

    print("Map saved:", map_path)
    display(qa_map)
else:
    print("No successfully labelled training trip is available for map inspection.")


In [ ]:

# 13. Plot assigned-distance distribution
if not summary_df.empty:
    plt.figure(figsize=(10, 4))
    for partition in ("train", "student_test"):
        values = summary_df.loc[
            summary_df["partition"] == partition, "p95_assigned_distance_m"
        ].dropna()
        if len(values):
            plt.hist(values, bins=20, alpha=0.55, label=partition)

    plt.axvline(
        MATCH_DISTANCE_THRESHOLD_M, linestyle="--",
        label=f"Match threshold ({MATCH_DISTANCE_THRESHOLD_M:.0f} m)"
    )
    plt.title("Per-trip 95th percentile distance to assigned route arc")
    plt.xlabel("Distance (metres)")
    plt.ylabel("Number of trips")
    plt.legend()
    plt.tight_layout()
    plt.savefig(LABEL_DIR / "p95_assigned_distance_distribution.png", dpi=160)
    plt.show()
else:
    print("No summary available to plot.")


In [ ]:

# 14. Verify generated files
print("Generated files under:", LABEL_DIR)
for path in sorted(LABEL_DIR.glob("*")):
    if path.is_file():
        print(f"{path.name:45s} {path.stat().st_size / 1024:10.2f} KB")

print("\\nMap files under:", MAP_DIR)
for path in sorted(MAP_DIR.glob("*.html")):
    print(path.name, f"({path.stat().st_size / 1024:.1f} KB)")



## 15. Output schema and handoff to teammates

Main files:

| File | Use |
|---|---|
| `train_gps_route_labels.csv` | Training examples. Use only rows with `is_matched == True` unless the modelling team explicitly handles unmatched rows. |
| `student_test_gps_route_labels.csv` | Development-stage comparison only. Do not use for model fitting or parameter tuning. |
| `label_generation_summary.csv` | Per-trip coverage, route continuity, and distance diagnostics. |
| `label_generation_errors.csv` | Trips that failed strict validation/alignment. Exists only when failures occur. |
| `label_check_<trip_id>.html` | Folium visual QA map. |

Important columns in the per-GPS-point files:

- `gps_row_index`: original zero-based row number in `.track`.
- `timestamp`: source timestamp in seconds.
- `longitude`, `latitude`: original GPS coordinate.
- `route_position`: zero-based position in the ordered `.route` sequence. This is the monotonic sequence index.
- `arc_id`: actual `.arcs` row ID corresponding to the selected route position. This is the route-arc class label.
- `from_node_id`, `to_node_id`: endpoints of the selected arc.
- `fraction_along_arc`: projected position along the segment, from 0 to 1.
- `distance_to_assigned_arc_m`: perpendicular/endpoint distance in metres to the selected segment.
- `is_matched`: whether distance is at or below `MATCH_DISTANCE_THRESHOLD_M`.
- `label_status`: `matched`, `beyond_distance_threshold`, or `invalid_gps_or_timestamp`.

### Teammate handoff rules

1. Tell the modelling teammate to train using **only `train_gps_route_labels.csv`** and normally filter to `is_matched == True`.
2. Preserve `route_position` for sequence-aware checks. Do not use it as the model class; use `arc_id` as the route-arc label.
3. Keep `student_test` labels separate and use them only for development-stage evaluation/comparison.
4. Never include evaluator-testing labels in model training, feature selection, or parameter tuning.
5. Share the `label_generation_summary.csv` and at least one HTML QA map along with the labels.
6. If a trip failed strict route continuity or has poor match coverage, report it rather than silently omitting it.

### Limitations to explain in the report/viva

- The supplied `.route` sequence provides an ordered reference path; the GPS-to-route assignment is generated by this notebook, not directly supplied row-by-row.
- The algorithm minimises total squared point-to-route-segment distance subject to a non-decreasing route-position constraint. It does not use a learned motion model or HMM.
- A large assigned distance indicates a potentially poor observation, route issue, or data anomaly; it is flagged rather than hidden.
- The 100 m threshold is a configurable QA rule, not a dataset-provided ground-truth tolerance. Review the distance distribution and maps before finalising it.
- The dynamic-programming implementation is exact but uses memory proportional to `number_of_valid_GPS_points × number_of_route_arcs`; the configured cell limit prevents unexpected memory exhaustion.
